# FastOTF2 Converter Benchmark

One-click comparison of the **Chapel** `fastotf2` converter against **Python** and **C**
converters, measuring the wall-clock time to convert OTF2 traces to tabular output.

| Tool | CSV | Parquet |
|------|-----|---------|
| Chapel (`fastotf2` container) | ✓ | ✓ |
| Python (`otf2` + pyarrow) | ✓ | ✓ |
| C (OTF2 C library) | ✓ | — (Arrow toolchain out of scope) |

All three converters ship inside **one Apptainer container** (`container/fastotf2-bench.sif`),
so they use an identical OTF2 version and the workflow is portable: the only host
requirements are Apptainer + SLURM and readable trace paths.

**To run everything: `Run All`.** The notebook will
1. build the container image if it is missing,
2. submit a single-node exclusive SLURM job that times every converter/format/trace,
3. load the results and render a comparison **table** + chart.

All configuration (traces, tool/format combos, repeats, image path, SLURM account/
partition) lives in [`bench.config.sh`](bench.config.sh) — edit that one file to retarget.

In [ ]:
# --- Bootstrap: read configuration from bench.config.sh (single source of truth) ---
import os, subprocess, time

REPO_DIR = os.getcwd()
CONFIG = os.path.join(REPO_DIR, "bench.config.sh")
assert os.path.exists(CONFIG), f"bench.config.sh not found in {REPO_DIR}; run this notebook from the repo root."

def cfg(var):
    """Read a scalar variable's value from bench.config.sh."""
    r = subprocess.run(
        ["bash", "-c", f'source "{CONFIG}" >/dev/null 2>&1; printf "%s" "${{{var}}}"'],
        capture_output=True, text=True,
    )
    return r.stdout.strip()

BENCH_SIF   = cfg("BENCH_SIF")
RESULTS_CSV = cfg("BENCH_RESULTS_CSV")
ACCOUNT     = cfg("BENCH_SLURM_ACCOUNT")
PARTITION   = cfg("BENCH_SLURM_PARTITION")

print("Repo:      ", REPO_DIR)
print("Image:     ", BENCH_SIF)
print("Results:   ", RESULTS_CSV)
print("SLURM:     ", f"account={ACCOUNT} partition={PARTITION}")
print("Traces / combos configured in bench.config.sh")

: 

In [ ]:
# --- Ensure the benchmark container image exists (build once, ~minutes) ---
if os.path.exists(BENCH_SIF):
    print("Bench image present:", BENCH_SIF)
else:
    print("Bench image missing — building it now (one-time)...")
    subprocess.run(["bash", os.path.join(REPO_DIR, "container", "build-bench-image.sh")],
                   cwd=REPO_DIR, check=True)
    print("Build complete:", BENCH_SIF)

In [ ]:
# --- Submit the benchmark to SLURM (1 exclusive node) and wait for completion ---
os.makedirs(os.path.join(REPO_DIR, "results", "logs"), exist_ok=True)

submit = subprocess.run(
    ["sbatch", f"--account={ACCOUNT}", f"--partition={PARTITION}", "--parsable",
     os.path.join(REPO_DIR, "benchmark", "run_benchmark.sbatch")],
    cwd=REPO_DIR, capture_output=True, text=True,
)
if submit.returncode != 0:
    raise RuntimeError(f"sbatch failed:\n{submit.stderr}")
JOB_ID = submit.stdout.strip().split(";")[0]
print(f"Submitted SLURM job {JOB_ID}. Waiting for it to finish...")

log_out = os.path.join(REPO_DIR, "results", "logs", f"slurm-{JOB_ID}.out")
last_state = None
while True:
    q = subprocess.run(["squeue", "-h", "-j", JOB_ID, "-o", "%T"], capture_output=True, text=True)
    state = q.stdout.strip()
    if not state:
        break  # job has left the queue -> finished
    if state != last_state:
        print(f"  [{time.strftime('%H:%M:%S')}] job {JOB_ID}: {state}", flush=True)
        last_state = state
    time.sleep(15)
print(f"Job {JOB_ID} finished.\n")

if os.path.exists(log_out):
    print("----- tail of SLURM log -----")
    print("".join(open(log_out).readlines()[-30:]))

In [ ]:
# --- Load results and build the comparison table ---
import pandas as pd

df = pd.read_csv(RESULTS_CSV)
if (df["status"] != 0).any():
    print("WARNING: some runs failed (status != 0):")
    print(df[df["status"] != 0][["trace", "tool", "format", "status"]].to_string(index=False))

# Average over repeats.
agg = (df[df["status"] == 0]
       .groupby(["trace", "tool", "format"], as_index=False)
       .agg(seconds=("seconds", "mean"), output_bytes=("output_bytes", "mean")))
agg["combo"] = agg["tool"].str.capitalize() + " (" + agg["format"] + ")"

col_order = [c for c in ["Chapel (CSV)", "Python (CSV)", "C (CSV)", "Chapel (PARQUET)", "Python (PARQUET)"]
             if c in set(agg["combo"])]
table = (agg.pivot(index="trace", columns="combo", values="seconds")
            .reindex(columns=col_order))
table.columns.name = None
print("Conversion time (seconds), averaged over repeats:")
table.round(2)

In [ ]:
# --- Speedup of Chapel and C over the Python baseline (per format) ---
sp = agg.pivot_table(index="trace", columns=["format", "tool"], values="seconds")
speedup = pd.DataFrame(index=table.index)
for fmt in sp.columns.get_level_values(0).unique():
    block = sp[fmt]
    if "python" not in block.columns:
        continue
    for tool in block.columns:
        if tool == "python":
            continue
        speedup[f"{tool.capitalize()} vs Python ({fmt})"] = block["python"] / block[tool]
print("Speedup factor over Python (higher = faster than Python):")
speedup.round(1)

In [ ]:
# --- Grouped bar chart of conversion time ---
import matplotlib.pyplot as plt

ax = table.plot(kind="bar", figsize=(11, 6), logy=True, width=0.8, edgecolor="black")
ax.set_ylabel("Conversion time (s, log scale)")
ax.set_xlabel("Trace")
ax.set_title("OTF2 conversion time by tool and format")
ax.legend(title="Tool (format)", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(REPO_DIR, "results", "benchmark_chart.png"), dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# --- Persist the comparison table (Markdown + CSV) ---
md_path  = os.path.join(REPO_DIR, "results", "benchmark_table.md")
csv_path = os.path.join(REPO_DIR, "results", "benchmark_table.csv")
table.round(2).to_csv(csv_path)
with open(md_path, "w") as f:
    f.write("# FastOTF2 converter benchmark\n\n")
    f.write("Conversion time in seconds (averaged over repeats).\n\n")
    f.write(table.round(2).to_markdown())
    f.write("\n\n## Speedup over Python\n\n")
    f.write(speedup.round(1).to_markdown())
    f.write("\n")
print("Wrote:", md_path)
print("Wrote:", csv_path)
print(open(md_path).read())

## Methodology

- Every conversion runs inside `container/fastotf2-bench.sif` on **one exclusive node**
  via SLURM, so timings are comparable and uncontended.
- Wall-clock time is measured around each converter invocation; output is written to
  scratch and deleted between runs. Repeats (`BENCH_REPEATS`) are averaged.
- Chapel uses all node cores (`CHPL_RT_NUM_THREADS_PER_LOCALE`) — its data-parallel
  advantage. Python and C are single-threaded, as shipped.
- C converts to CSV only; Parquet in C requires the Apache Arrow C++/GLib toolchain and
  is out of scope for the baseline.

Raw per-run data is in `results/results.csv`.